# Project 1 — Robust Console Calculator

**Goal:** Build a four-operation calculator with reusable logic, safe numeric conversion, and friendly error handling. The examples demonstrate normal operations; optional interactive input is commented out so Run All works unattended.

**Level:** Basics · Project · **Study time:** 2–3 hours · **Prerequisite:** lessons 00–14

## You will learn

- Separate calculation from input/output
- Handle invalid operators and division by zero
- Test boundary cases and format results

The runnable implementation is a worked example. Submit your own changes and regression tests in the independent section.

In [ ]:
# Setup: run this first. On Google Colab (or anywhere the course files are missing) it
# downloads the course and moves into this notebook's folder. Elsewhere it does nothing.
import os, sys
if not os.path.exists("../README.md"):
    course = os.path.join(os.path.expanduser("~"), "python-notebook")
    if not os.path.isdir(course):
        os.system(f"git clone -q --depth 1 https://github.com/SwarmLadder/python-notebook {course}")
    os.chdir(os.path.join(course, "basics", "projects"))
    sys.path.insert(0, os.getcwd())

## Examples

### 1. Requirements and design

Write a pure function `calculate(a, operator, b)` that supports addition, subtraction, multiplication, and division. Invalid operators should raise ValueError; dividing by zero should raise ZeroDivisionError. This allows callers to handle exceptions.

In [ ]:
def calculate(a, operator, b):
    if operator == "+":
        return a + b
    if operator == "-":
        return a - b
    if operator == "*":
        return a * b
    if operator == "/":
        if b == 0:
            raise ZeroDivisionError("cannot divide by zero")
        return a / b
    raise ValueError(f"unsupported operator: {operator}")

**Check your understanding:** Why is error handling easier when a function returns a value?

### 2. Testing the computation

Test each operator plus a failure case. Use assertions to make the expectations executable and to guard against accidental changes.

In [ ]:
assert calculate(3, "+", 4) == 7
assert calculate(3, "-", 4) == -1
assert calculate(3, "*", 4) == 12
assert calculate(9, "/", 3) == 3
try:
    calculate(1, "/", 0)
except ZeroDivisionError:
    print("Zero division handled correctly")

**Check your understanding:** What additional test would reveal a negative-number bug?

### 3. Safe text parsing and display

A command-line program receives text. Convert numbers inside a try block, then call the pure calculation function. The example runs with a fixed simulated request; optionally uncomment `input` calls to make it interactive.

In [ ]:
def process_request(left_text, operator, right_text):
    try:
        left = float(left_text)
        right = float(right_text)
        result = calculate(left, operator, right)
    except (ValueError, ZeroDivisionError) as error:
        return f"Error: {error}"
    return f"Result: {result:g}"

print(process_request("12", "+", "3"))
print(process_request("12", "/", "0"))
print(process_request("abc", "*", "3"))
# print(process_request(input("First: "), input("Operator: "), input("Second: ")))

**Check your understanding:** Why keep the interactive input separate from calculate?

## Independent project work — Order Desk calculator ticket

The calculator already handles four operations. Support asks for powers (`**`) and remainder (`%`), and for an operation history that survives a failed request. Modify the **existing** `calculate` function without removing its old behavior.

**Acceptance criteria.** `calculate(7, '%', 3)` returns `1`; dividing or taking remainder by zero raises `ZeroDivisionError`. Implement `calculate_and_record(a, operator, b, history)` to return the numeric result and append `(a, operator, b, result)` to the caller-supplied history **only after success**. Invalid operators propagate `ValueError`. A failed request must leave history unchanged. Use separate history lists for separate callers.

**Regression ticket.** After one successful purchase calculation, an invalid request must not erase that history. Reproduce the failure, fix it and explain why recording a failure would mislead support. Complete the provided acceptance check after your implementation.

**Exercise 1 — Feature.** Extend `calculate` to support exponentiation `**` without changing addition, subtraction, multiplication or division.

In [ ]:
# TODO: Exercise 1
# Explain the steps, then implement and test your answer.

**Exercise 2 — Safety.** Support remainder `%` while ensuring `7 % 3` is `1` and a zero divisor raises `ZeroDivisionError`.

In [ ]:
# TODO: Exercise 2
# Explain the steps, then implement and test your answer.

**Exercise 3 — Regression fix.** Implement `calculate_and_record` below. Its `history` argument is an existing list owned by the caller; never change it when calculation raises an exception.

In [ ]:
def calculate_and_record(a, operator, b, history):
    """Return a calculation and append (a, operator, b, result) only on success."""
    raise NotImplementedError  # Implement the ticket here.

**Exercise 4 — Acceptance check.** Run the supplied checks after implementing the ticket. Add at least three of your own tests for negative numbers, an empty history, or unexpected operators. Explain one design choice in your review.

In [ ]:
def verify_calculator_ticket():
    assert calculate(2, "**", 3) == 8, "Keep exponentiation working."
    assert calculate(7, "%", 3) == 1, "Remainder must work."
    for operation in ("/", "%"):
        try:
            calculate(7, operation, 0)
        except ZeroDivisionError:
            pass
        else:
            raise AssertionError("A zero divisor must fail for both operations.")

    history = []
    assert calculate_and_record(5, "+", 3, history) == 8
    assert history == [(5, "+", 3, 8)], "Record every successful result once."
    previous = history[:]
    for a, operation, b, error in (
        (1, "/", 0, ZeroDivisionError),
        (1, "%", 0, ZeroDivisionError),
        (1, "?", 2, ValueError),
    ):
        try:
            calculate_and_record(a, operation, b, history)
        except error:
            pass
        else:
            raise AssertionError("Do not hide invalid operations or zero division.")
        assert history == previous, "A failure must preserve previous history."

    other = []
    assert calculate_and_record(7, "%", 3, other) == 1
    assert other == [(7, "%", 3, 1)]
    assert history == previous, "Different callers own separate histories."

try:
    verify_calculator_ticket()
except NotImplementedError:
    print("Project check: Not attempted — implement the history function.")
except Exception as error:
    print(f"Project check: Needs improvement — {type(error).__name__}: {error}")
else:
    print("Project check: Passed — all supplied acceptance cases.")